In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import open3d as o3d
import numpy as np

class PointNetBackbone(nn.Module):
    def __init__(self, global_feature_dim=1024):
        super(PointNetBackbone, self).__init__()
        # MLP layers implemented as 1D convolutions (kernel_size=1)
        self.conv1 = nn.Conv1d(3, 64, kernel_size=1)
        self.bn1 = nn.BatchNorm1d(64)
        
        self.conv2 = nn.Conv1d(64, 128, kernel_size=1)
        self.bn2 = nn.BatchNorm1d(128)
        
        self.conv3 = nn.Conv1d(128, global_feature_dim, kernel_size=1)
        self.bn3 = nn.BatchNorm1d(global_feature_dim)
        
    def forward(self, x):
        """
        Input:
            x: tensor of shape [B, N, 3] where B is batch size, N is number of points.
        Output:
            global_feature: tensor of shape [B, global_feature_dim]
        """
        # Transpose to [B, 3, N] for 1D convolution
        x = x.transpose(2, 1)
        x = F.relu(self.bn1(self.conv1(x)))   # [B, 64, N]
        x = F.relu(self.bn2(self.conv2(x)))   # [B, 128, N]
        x = F.relu(self.bn3(self.conv3(x)))   # [B, global_feature_dim, N]
        # Perform max pooling across points (dim=2)
        global_feature = torch.max(x, dim=2)[0] # [B, global_feature_dim]
        return global_feature

# Example function to compute cosine similarity between two point clouds
def compute_similarity(pointnet, global_feature1, pcd2):
    """
    Computes the cosine similarity between two point clouds using the PointNet backbone.
    
    Args:
        pointnet: an instance of PointNetBackbone.
        pc1, pc2: input point clouds of shape [B, N, 3].
        
    Returns:
        similarity: tensor of shape [B] with cosine similarity scores.
    """
    # Compute global features for each point cloud
    global_feat2 = pointnet(pcd2)
    # Compute cosine similarity along feature dimension
    similarity = F.cosine_similarity(global_feature1, global_feat2, dim=1)
    return similarity

def prepare_pcd(pcd):
    """
    Prepare the point cloud data for input to the PointNet backbone.
    
    Args:
        pcd: Open3D point cloud object.
        
    Returns:
        pc: tensor of shape [B, N, 3] where B is batch size, N is number of points.
    """
    # Convert Open3D point cloud to numpy array
    points = np.asarray(pcd.points)
    # Convert numpy array to tensor
    pc = torch.tensor(points, dtype=torch.float32)
    # Add batch dimension
    pc = pc.unsqueeze(0)
    return pc

# Example usage:
if __name__ == "__main__":
    # load point clouds
    scan = o3d.io.read_point_cloud("gedi_data/working_data/scan/scan_best/point_cloud/763620_9.ply")
    cad1 = o3d.io.read_point_cloud("gedi_data/working_data/cad/point_cloud/763620.ply")
    cad2 = o3d.io.read_point_cloud("gedi_data/working_data/cad/point_cloud/763621.ply")
    cad3 = o3d.io.read_point_cloud("gedi_data/working_data/cad/point_cloud/763638.ply")
    
    scan = prepare_pcd(scan)
    pc1 = prepare_pcd(cad1)
    pc2 = prepare_pcd(cad2)
    pc3 = prepare_pcd(cad3)
    
    # Initialize the PointNet backbone
    pointnet = PointNetBackbone(global_feature_dim=1024)
    
    # compute global feature for the first point cloud
    global_feature1 = pointnet(scan)
    
    for pcd in [pc1, pc2, pc3]:
        similarity_scores = compute_similarity(pointnet, global_feature1, pcd)
        print("Name : ", pcd)
        print("Cosine similarity scores:", similarity_scores)

Cosine similarity scores: tensor([0.7012], grad_fn=<DivBackward0>)
Cosine similarity scores: tensor([0.7012], grad_fn=<DivBackward0>)
Cosine similarity scores: tensor([0.7219], grad_fn=<DivBackward0>)
